E-Commerce Conversion Prediction

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, classification_report

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 220)

SEED = 42
N_FOLDS = 5
SEEDS = [42, 7, 2024]   
print('lgb:', lgb.__version__, ' xgb:', xgb.__version__)

lgb: 4.6.0  xgb: 3.2.0


load the data

In [2]:
train = pd.read_csv('train.csv')
public = pd.read_csv('public_test.csv')
private = pd.read_csv('private_test.csv')
sample = pd.read_csv('sample_submission.csv')

print(train.shape, public.shape, private.shape, sample.shape)
train.head()

(10000, 14) (3000, 14) (3000, 13) (3000, 2)


,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
0,1,58.0,103593.708812,2,Mobile,Organic,5,4,9.61,3,0,11,2418,0
1,2,26.0,36451.716984,2,Mobile,Social Media,11,3,17.63,2,0,14,1213,0
2,3,19.0,30511.228700,3,Mobile,Referral,1,1,13.25,5,0,5,2849,0
3,4,48.0,87789.172342,3,Mobile,Email,14,12,NaN,1,1,19,7610,0
4,5,35.0,105229.249067,2,Mobile,Social Media,14,21,16.92,1,0,5,9261,0


EDA

In [3]:
# how balanced is the target?
print('train positive rate :', train['Converted'].mean().round(4))
print('public positive rate:', public['Converted'].mean().round(4))

train positive rate : 0.3087
public positive rate: 0.2953


In [4]:
# missing values
miss = train.isna().sum()
miss[miss > 0]

Age             1480
Income           984
Time_On_Site    1848
dtype: int64

In [5]:
train.describe().T

,count,mean,std,min,25%,50%,75%,max
User_ID,10000.0,5000.500000,2886.895680,1.0,2500.750000,5000.500000,7500.250000,10000.000000
Age,8520.0,41.457746,13.770164,18.0,29.000000,41.000000,53.000000,65.000000
Income,9016.0,69961.772797,24790.673822,12000.0,52294.644359,70171.613672,86907.747154,161687.774167
City_Tier,10000.0,1.933400,0.737712,1.0,1.000000,2.000000,2.000000,3.000000
Pages_Viewed,10000.0,15.608500,8.623460,1.0,8.000000,16.000000,23.000000,30.000000
Products_Viewed,10000.0,15.219300,8.927563,1.0,8.000000,15.000000,23.000000,37.000000
Time_On_Site,8152.0,13.667241,19.438244,0.8,7.640000,11.145000,15.630000,607.390000
Previous_Purchases,10000.0,2.972500,1.738691,0.0,2.000000,3.000000,4.000000,12.000000
Discount_Seen,10000.0,0.540700,0.498366,0.0,0.000000,1.000000,1.000000,1.000000
Browser_Version,10000.0,12.473900,6.892856,1.0,7.000000,12.000000,18.000000,24.000000


In [ ]:
for c in ['City_Tier', 'Device_Type', 'Traffic_Source', 'Discount_Seen']:
    g = train.groupby(c)['Converted'].agg(['mean','count'])
    print(c)
    print(g, '\n')

City_Tier
               mean  count
City_Tier                 
1          0.302991   3076
2          0.320337   4514
3          0.294191   2410 

Device_Type
                 mean  count
Device_Type                 
Desktop      0.330906   2472
Mobile       0.300629   6523
Tablet       0.306468   1005 

Traffic_Source
                    mean  count
Traffic_Source                 
Email           0.349359    936
Organic         0.291762   3071
Paid Ads        0.277123   2544
Referral        0.381181   1456
Social Media    0.303061   1993 

Discount_Seen
                   mean  count
Discount_Seen                 
0              0.255171   4593
1              0.354171   5407 



In [ ]:
(train['Income'] == 12000.0).sum(), (train['Income'] == 12000.0).mean()

(np.int64(73), np.float64(0.0073))

feature engineering

In [ ]:
def add_features(df):
    df = df.copy()

    # missing flags
    df['age_missing'] = df['Age'].isna().astype(int)
    df['income_missing'] = df['Income'].isna().astype(int)
    df['time_missing'] = df['Time_On_Site'].isna().astype(int)
    df['n_missing'] = df[['Age','Income','Time_On_Site']].isna().sum(axis=1)

    df['income_floor'] = (df['Income'] == 12000.0).astype(int)

    
    inc_med = df['Income'].median()
    time_med = df['Time_On_Site'].median()
    age_med = df['Age'].median()

    df['log_income'] = np.log1p(df['Income'].fillna(inc_med))
    df['log_time'] = np.log1p(df['Time_On_Site'].fillna(time_med))

    pages = df['Pages_Viewed'].astype(float)
    products = df['Products_Viewed'].astype(float)
    t = df['Time_On_Site'].fillna(time_med).astype(float)
    age = df['Age'].fillna(age_med).astype(float)
    inc = df['Income'].fillna(inc_med).astype(float)
    prev = df['Previous_Purchases'].astype(float)
    disc = df['Discount_Seen'].astype(float)

    df['products_per_page'] = products / (pages + 1)
    df['pages_minus_products'] = pages - products
    df['time_per_page'] = t / (pages + 1)
    df['time_per_product'] = t / (products + 1)
    df['engagement'] = pages + products

    
    df['engagement_x_discount'] = df['engagement'] * disc
    df['prev_x_discount'] = prev * disc
    df['pages_x_discount'] = pages * disc
    df['products_x_discount'] = products * disc
    df['log_time_x_discount'] = df['log_time'] * disc

    # demographic-ish
    df['age_x_income'] = age * np.log1p(inc) / 100
    df['prev_per_age'] = prev / (age + 1)

    # binary flags
    df['high_engagement'] = ((pages > 20) & (products > 20)).astype(int)
    df['low_engagement'] = ((pages < 5) & (products < 5)).astype(int)
    df['time_outlier'] = (t > 100).astype(int)

    # buckets
    df['age_bucket'] = pd.cut(df['Age'].fillna(-1),
                              bins=[-2, 0, 25, 35, 45, 55, 100],
                              labels=[-1,0,1,2,3,4]).astype(int)
    df['income_bucket'] = pd.qcut(df['Income'].fillna(inc_med),
                                  q=10, labels=False, duplicates='drop')
    df['Campaign_Bucket'] = (df['Campaign_Code'] // 1000).astype(int)

    return df

In [9]:
train = add_features(train)
public = add_features(public)
private = add_features(private)

print(train.shape, public.shape, private.shape)

(10000, 39) (3000, 39) (3000, 38)


In [ ]:
CAT_COLS = ['Device_Type', 'Traffic_Source']

for c in CAT_COLS:
    all_vals = pd.concat([train[c], public[c], private[c]]).astype(str)
    cats = pd.Index(all_vals.unique())
    train[c] = pd.Categorical(train[c].astype(str), categories=cats)
    public[c] = pd.Categorical(public[c].astype(str), categories=cats)
    private[c] = pd.Categorical(private[c].astype(str), categories=cats)
    
DROP = ['User_ID', 'Converted']
FEATURES = [c for c in train.columns if c not in DROP]
print(len(FEATURES), 'features')
FEATURES

37

 features


['Age',
 'Income',
 'City_Tier',
 'Device_Type',
 'Traffic_Source',
 'Pages_Viewed',
 'Products_Viewed',
 'Time_On_Site',
 'Previous_Purchases',
 'Discount_Seen',
 'Browser_Version',
 'Campaign_Code',
 'age_missing',
 'income_missing',
 'time_missing',
 'n_missing',
 'income_floor',
 'log_income',
 'log_time',
 'products_per_page',
 'pages_minus_products',
 'time_per_page',
 'time_per_product',
 'engagement',
 'engagement_x_discount',
 'prev_x_discount',
 'pages_x_discount',
 'products_x_discount',
 'log_time_x_discount',
 'age_x_income',
 'prev_per_age',
 'high_engagement',
 'low_engagement',
 'time_outlier',
 'age_bucket',
 'income_bucket',
 'Campaign_Bucket']

In [11]:
X = train[FEATURES]
y = train['Converted'].astype(int)
X_pub = public[FEATURES]
y_pub = public['Converted'].astype(int)
X_prv = private[FEATURES]
print(X.shape, X_pub.shape, X_prv.shape)

(10000, 37) (3000, 37) (3000, 37)


Modelling

In [12]:
def find_best_threshold(y_true, proba):
    # just sweep, this is fine - 360 evals takes nothing
    best_t = 0.5
    best_f1 = -1
    for t in np.linspace(0.05, 0.95, 361):
        pred = (proba >= t).astype(int)
        f = f1_score(y_true, pred)
        if f > best_f1:
            best_f1 = f
            best_t = float(t)
    return best_t, best_f1

In [13]:
def run_lgb(X, y, X_pub, X_prv, seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    pub = np.zeros(len(X_pub))
    prv = np.zeros(len(X_prv))

    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'learning_rate': 0.025,
        'num_leaves': 47,
        'min_data_in_leaf': 50,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.85,
        'bagging_freq': 5,
        'lambda_l1': 0.5,
        'lambda_l2': 2.0,
        'verbose': -1,
        'seed': seed,
    }

    for fold, (tr, va) in enumerate(skf.split(X, y)):
        dtr = lgb.Dataset(X.iloc[tr], y.iloc[tr], categorical_feature=CAT_COLS)
        dva = lgb.Dataset(X.iloc[va], y.iloc[va], categorical_feature=CAT_COLS, reference=dtr)
        m = lgb.train(params, dtr,
                      num_boost_round=6000,
                      valid_sets=[dva],
                      callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)])
        oof[va] = m.predict(X.iloc[va], num_iteration=m.best_iteration)
        pub += m.predict(X_pub, num_iteration=m.best_iteration) / N_FOLDS
        prv += m.predict(X_prv, num_iteration=m.best_iteration) / N_FOLDS
    return oof, pub, prv

In [14]:
def run_xgb(X, y, X_pub, X_prv, seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    pub = np.zeros(len(X_pub))
    prv = np.zeros(len(X_prv))

    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'eta': 0.025,
        'max_depth': 5,
        'min_child_weight': 8,
        'subsample': 0.85,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.5,
        'reg_lambda': 2.0,
        'tree_method': 'hist',
        'enable_categorical': True,
        'random_state': seed,
        'verbosity': 0,
    }

    for tr, va in skf.split(X, y):
        dtr = xgb.DMatrix(X.iloc[tr], label=y.iloc[tr], enable_categorical=True)
        dva = xgb.DMatrix(X.iloc[va], label=y.iloc[va], enable_categorical=True)
        dpub = xgb.DMatrix(X_pub, enable_categorical=True)
        dprv = xgb.DMatrix(X_prv, enable_categorical=True)

        m = xgb.train(params, dtr,
                      num_boost_round=6000,
                      evals=[(dva,'v')],
                      early_stopping_rounds=200,
                      verbose_eval=False)
        rng = (0, m.best_iteration + 1)
        oof[va] = m.predict(dva, iteration_range=rng)
        pub += m.predict(dpub, iteration_range=rng) / N_FOLDS
        prv += m.predict(dprv, iteration_range=rng) / N_FOLDS
    return oof, pub, prv

In [ ]:
def run_cat(X, y, X_pub, X_prv, seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    pub = np.zeros(len(X_pub))
    prv = np.zeros(len(X_prv))

    cat_idx = [X.columns.get_loc(c) for c in CAT_COLS]

    def prep(df):
        df = df.copy()
        for c in CAT_COLS:
            df[c] = df[c].astype(str)
        return df

    Xc = prep(X)
    Xp = prep(X_pub)
    Xr = prep(X_prv)

    for tr, va in skf.split(Xc, y):
        tr_pool = Pool(Xc.iloc[tr], y.iloc[tr], cat_features=cat_idx)
        va_pool = Pool(Xc.iloc[va], y.iloc[va], cat_features=cat_idx)
        m = CatBoostClassifier(iterations=4000,
                               learning_rate=0.03,
                               depth=6,
                               l2_leaf_reg=5.0,
                               random_seed=seed,
                               eval_metric='Logloss',
                               early_stopping_rounds=200,
                               verbose=False)
        m.fit(tr_pool, eval_set=va_pool)
        oof[va] = m.predict_proba(va_pool)[:,1]
        pub += m.predict_proba(Pool(Xp, cat_features=cat_idx))[:,1] / N_FOLDS
        prv += m.predict_proba(Pool(Xr, cat_features=cat_idx))[:,1] / N_FOLDS
    return oof, pub, prv

In [16]:
t0 = time.time()
results = {}
for name, fn in [('lgb', run_lgb), ('xgb', run_xgb), ('cat', run_cat)]:
    oof_sum = np.zeros(len(X))
    pub_sum = np.zeros(len(X_pub))
    prv_sum = np.zeros(len(X_prv))
    for s in SEEDS:
        oo, pp, rr = fn(X, y, X_pub, X_prv, seed=s)
        oof_sum += oo / len(SEEDS)
        pub_sum += pp / len(SEEDS)
        prv_sum += rr / len(SEEDS)
        print(f'  {name} seed={s} oof_auc={roc_auc_score(y, oo):.4f} pub_auc={roc_auc_score(y_pub, pp):.4f}')
    results[name] = (oof_sum, pub_sum, prv_sum)

print(f'\ntotal time: {time.time()-t0:.1f}s')

Training until validation scores don't improve for 200 rounds

Early stopping, best iteration is:
[134]	valid_0's binary_logloss: 0.55245


Training until validation scores don't improve for 200 rounds


Early stopping, best iteration is:
[99]	valid_0's binary_logloss: 0.551846


Training until validation scores don't improve for 200 rounds


...

  xgb seed=7 oof_auc=0.7240 pub_auc=0.7305


  xgb seed=2024 oof_auc=0.7264 pub_auc=0.7309


  cat seed=42 oof_auc=0.7286 pub_auc=0.7350


  cat seed=7 oof_auc=0.7278 pub_auc=0.7351


  cat seed=2024 oof_auc=0.7286 pub_auc=0.7358

total time: 1222.0s


In [ ]:
oof_lgb, pub_lgb, prv_lgb = results['lgb']
oof_xgb, pub_xgb, prv_xgb = results['xgb']
oof_cat, pub_cat, prv_cat = results['cat']

w1 = w2 = w3 = 1/3
oof = w1*oof_lgb + w2*oof_xgb + w3*oof_cat
pub_proba = w1*pub_lgb + w2*pub_xgb + w3*pub_cat
prv_proba = w1*prv_lgb + w2*prv_xgb + w3*prv_cat

print('individual:')
for nm,(oo,pp,_) in results.items():
    print(f'  {nm}  oof_auc={roc_auc_score(y,oo):.5f}  pub_auc={roc_auc_score(y_pub,pp):.5f}')
print(f'\nblend  oof_auc={roc_auc_score(y,oof):.5f}  pub_auc={roc_auc_score(y_pub,pub_proba):.5f}')

individual:
  lgb  oof_auc=0.72379  pub_auc=0.72904
  xgb  oof_auc=0.72658  pub_auc=0.73096
  cat  oof_auc=0.72935  pub_auc=0.73557

blend  oof_auc=0.72784  pub_auc=0.73288


In [ ]:
oof_t, oof_f1 = find_best_threshold(y.values, oof)
pub_t, pub_f1 = find_best_threshold(y_pub.values, pub_proba)
print(f'oof: best t={oof_t:.3f}  f1={oof_f1:.5f}')
print(f'pub: best t={pub_t:.3f}  f1={pub_f1:.5f}')

oof: best t=0.270  f1=0.57343
pub: best t=0.250  f1=0.55569


In [ ]:
rows = []
for t in np.arange(0.20, 0.42, 0.02):
    rows.append({
        't': round(float(t),3),
        'oof_f1': f1_score(y, (oof>=t).astype(int)),
        'pub_f1': f1_score(y_pub, (pub_proba>=t).astype(int)),
        'oof_pos_rate': float((oof>=t).mean()),
        'pub_pos_rate': float((pub_proba>=t).mean()),
    })
pd.DataFrame(rows).round(4)

,t,oof_f1,pub_f1,oof_pos_rate,pub_pos_rate
0,0.20,0.5547,0.5476,0.6608,0.6493
1,0.22,0.5608,0.5518,0.6189,0.6120
2,0.24,0.5666,0.5528,0.5850,0.5730
3,0.26,0.5707,0.5481,0.5503,0.5390
4,0.28,0.5719,0.5434,0.5198,0.5033
5,0.30,0.5697,0.5430,0.4851,0.4793
6,0.32,0.5661,0.5407,0.4527,0.4457
7,0.34,0.5584,0.5431,0.4216,0.4130
8,0.36,0.5480,0.5322,0.3899,0.3837
9,0.38,0.5362,0.5244,0.3575,0.3530


In [ ]:
pub_pred = (pub_proba >= oof_t).astype(int)
print(f'public f1 at threshold {oof_t:.3f}: {f1_score(y_pub, pub_pred):.5f}')
print(classification_report(y_pub, pub_pred, digits=4))

public f1 at threshold 0.270: 0.54709
              precision    recall  f1-score   support

           0     0.8500    0.5762    0.6868      2114
           1     0.4282    0.7573    0.5471       886

    accuracy                         0.6297      3000
   macro avg     0.6391    0.6667    0.6169      3000
weighted avg     0.7254    0.6297    0.6455      3000



Refit on train + public_test

In [ ]:
X_full = pd.concat([X, X_pub], ignore_index=True)
y_full = pd.concat([y, y_pub], ignore_index=True)
print('full:', X_full.shape, y_full.shape)


dummy = X_full.iloc[:1].copy()

f_lgb_oof = np.zeros(len(X_full))
f_xgb_oof = np.zeros(len(X_full))
f_cat_oof = np.zeros(len(X_full))
f_lgb_prv = np.zeros(len(X_prv))
f_xgb_prv = np.zeros(len(X_prv))
f_cat_prv = np.zeros(len(X_prv))

t0 = time.time()
for s in SEEDS:
    oo, _, rr = run_lgb(X_full, y_full, dummy, X_prv, seed=s); f_lgb_oof += oo/len(SEEDS); f_lgb_prv += rr/len(SEEDS)
    oo, _, rr = run_xgb(X_full, y_full, dummy, X_prv, seed=s); f_xgb_oof += oo/len(SEEDS); f_xgb_prv += rr/len(SEEDS)
    oo, _, rr = run_cat(X_full, y_full, dummy, X_prv, seed=s); f_cat_oof += oo/len(SEEDS); f_cat_prv += rr/len(SEEDS)
print(f'refit time: {time.time()-t0:.1f}s')

full: (13000, 37) (13000,)


Training until validation scores don't improve for 200 rounds


Early stopping, best iteration is:
[110]	valid_0's binary_logloss: 0.542124


Training until validation scores don't improve for 200 rounds


Early stopping, best iteration is:
[110]	valid_0's binary_logloss: 0.553562


...

Training until validation scores don't improve for 200 rounds


Early stopping, best iteration is:
[94]	valid_0's binary_logloss: 0.551298


Training until validation scores don't improve for 200 rounds


Early stopping, best iteration is:
[88]	valid_0's binary_logloss: 0.554741


refit time: 1323.8s


In [22]:
oof_full = (f_lgb_oof + f_xgb_oof + f_cat_oof) / 3
prv_final = (f_lgb_prv + f_xgb_prv + f_cat_prv) / 3

final_t, final_f1 = find_best_threshold(y_full.values, oof_full)
print(f'full-data OOF auc = {roc_auc_score(y_full, oof_full):.5f}')
print(f'full-data OOF f1  = {final_f1:.5f}  at t={final_t:.3f}')

full-data OOF auc = 0.72964
full-data OOF f1  = 0.56629  at t=0.275


Build submission

In [23]:
preds = (prv_final >= final_t).astype(int)
print(f'positives in private predictions: {preds.sum()} / {len(preds)} ({preds.mean()*100:.2f}%)')

submission = pd.DataFrame({
    'User_ID': private['User_ID'].astype(int),
    'Converted': preds.astype(int),
})
submission.to_csv('submission.csv', index=False)
print('submission.csv shape', submission.shape)
submission.head()

positives in private predictions: 1530 / 3000 (51.00%)
submission.csv shape (3000, 2)


,User_ID,Converted
0,103001,0
1,103002,0
2,103003,1
3,103004,1
4,103005,0


In [ ]:
sub_check = pd.read_csv('submission.csv')
sample_check = pd.read_csv('sample_submission.csv')

assert list(sub_check.columns) == ['User_ID', 'Converted']
assert sub_check.shape == sample_check.shape
assert sub_check['User_ID'].equals(sample_check['User_ID'])
assert sub_check['Converted'].isin([0,1]).all()
print('format OK')

format OK
